In [1]:
!pip install -U sentence-transformers gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: gradio-client
    Found existing installation: gradio_client 2.6.1
    Uninstalling gradio_client-2.6.1:
      Successfully uninstalled gradio_client-2.6.1
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0
  Attempting uninstall: gradio
    Found existing installation: gradio 6.26.0
    Uninstalling gradio-6.26.0:
      Successfully uninstalled gradio-6.26.0


# Step 5.1: retrieval demo

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

ROOT = "/content/drive/MyDrive/sanskrit_retrieval"
gita = pd.read_csv(f"{ROOT}/data/gita_clean.csv", dtype={"id": str})

models = {
    "base": SentenceTransformer("intfloat/multilingual-e5-small", device="cuda"),
    "fine-tuned": SentenceTransformer(f"{ROOT}/models/e5_ft_v1/final", device="cuda"),
}

def enc(m, texts, prefix):
    return m.encode([prefix + t for t in texts], normalize_embeddings=True,
                    batch_size=64, show_progress_bar=False)

index = {(name, side): enc(m, gita[col].tolist(), "passage: ")
         for name, m in models.items()
         for side, col in [("english", "english"), ("sanskrit", "deva")]}

def search(query, model="fine-tuned", side="english", k=5):
    q = enc(models[model], [query], "query: ")[0]
    sims = index[(model, side)] @ q
    return [(gita.id[i], float(sims[i]), gita.deva[i], gita.english[i])
            for i in np.argsort(-sims)[:k]]

Mounted at /content/drive


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
tests = [
    "What does this verse say about karma?",
    "karmanyevadhikaraste ma phalesu kadacana",       # ASCII, verse 2.47
    "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन",                 # Devanagari, 2.47
    "the soul is eternal and cannot be killed",
    "how do I control a restless mind?",
    "who should I not share this teaching with?",
]

lines = []
for q in tests:
    lines.append(f"\n### Query: {q}")
    for name in ["base", "fine-tuned"]:
        lines.append(f"**{name}**")
        for vid, s, _, eng in search(q, name, "english", 3):
            lines.append(f"- {vid} ({s:.3f}): {eng[:110]}")
print("\n".join(lines))

os.makedirs(f"{ROOT}/results", exist_ok=True) if (os := __import__("os")) else None
open(f"{ROOT}/results/sample_outputs.md", "w", encoding="utf-8").write("\n".join(lines))


### Query: What does this verse say about karma?
**base**
- 2.47 (0.867): O ARJUNA, always remember what I am about to say to you for it is the law of KARMA, a law that one should alwa
- 4.16 (0.867): Even the wise men are confused about Karma (action) and Akarma (inaction). I shall now explain to you the trut
- 4.14 (0.866): The Lord continued: O Arjuna, since I, the immortal Lord, have no cravings for the fruits of action, I do not 
**fine-tuned**
- 4.16 (0.398): Even the wise men are confused about Karma (action) and Akarma (inaction). I shall now explain to you the trut
- 2.47 (0.392): O ARJUNA, always remember what I am about to say to you for it is the law of KARMA, a law that one should alwa
- 18.13 (0.371): O Mighty-armed Arjuna, learn and realize from Me the five causes of all the actions in this world as stated in

### Query: karmanyevadhikaraste ma phalesu kadacana
**base**
- 4.12 (0.828): O Arjuna, those who do actions or Karma for the fulfillment of their desires and goal

5011

In [4]:
import gradio as gr

def ui(query, model, side, k):
    if not query.strip():
        return "Enter a query."
    out = [f"**Verse {vid}** (score {s:.3f})\n\n{deva}\n\n{eng}\n\n---"
           for vid, s, deva, eng in search(query, model, side, int(k))]
    return "\n".join(out)

demo = gr.Interface(
    fn=ui,
    inputs=[gr.Textbox(label="Query (English, Devanagari or ASCII Sanskrit)"),
            gr.Radio(["fine-tuned", "base"], value="fine-tuned", label="Model"),
            gr.Radio(["english", "sanskrit"], value="english", label="Search in"),
            gr.Slider(1, 10, value=5, step=1, label="Top-k")],
    outputs=gr.Markdown(),
    title="Bhagavad Gita Sanskrit-English retrieval")
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://708fe62b0be7af73fa.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Quick check to run

In [8]:
def rank_of(query, gold, model, side="english"):
    q = enc(models[model], [query], "query: ")[0]
    sims = index[(model, side)] @ q
    g = gita.index[gita.id == gold][0]
    return int((sims > sims[g]).sum()) + 1

r = gita[gita.id == "2.47"].iloc[0]
variants = {
    "ascii half-line":  "karmanyevadhikaraste ma phalesu kadacana",
    "ascii full verse": r["ascii"],
    "deva half-line":   "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन",
    "deva full verse":  r["deva"],
}
for name, qt in variants.items():
    print(f"{name:18} base rank {rank_of(qt, '2.47', 'base'):>3} | ft rank {rank_of(qt, '2.47', 'fine-tuned'):>3}")

ascii half-line    base rank 131 | ft rank   4
ascii full verse   base rank 368 | ft rank   3
deva half-line     base rank 314 | ft rank   8
deva full verse    base rank 446 | ft rank   7
